# Session analysis, hand rehabilitation device

Basil Toufexis, MXEN4000 / MXEN4004, Curtin University.

Pick a save from the dropdown, then run the cells under it. Each cell does one
thing and prints its own result directly beneath, so the working stays visible
instead of arriving as one block from a single function.

A **game** is one folder, one block of one mode. Games by the same person on the
same day make up a **session**.

Force needs one warning up front. Each sensor pad reads a different number of
counts for the same real force, so raw counts are not comparable between
fingers. Where the session recorded a calibration, the sections divide each
finger by its own calibration press and say so. Where it did not, they say that
instead of quietly pooling the two.

## Setup

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import pandas as pd

from rehab_analysis import (
    build_catalogue, check, menu_options, prepare, use_style,
    sec_calibration, sec_overview, sec_quality, sec_compare,
    sec_reaction_time, sec_accuracy, sec_force, sec_individuation,
    sec_rhythm, sec_bilateral, sec_raw, sec_onset, sec_objective_one,
    sec_exclusions, sec_phase, sec_threshold_audit, sec_cue_modality,
    sec_dose, sec_sampling_note, sec_participant_progress,
)

use_style()

### Check the setup

The packages, and that the sessions folder is where the notebook expects it,
before anything tries to read data.

In [ ]:
ok = check()

## What is on disk

One row per game, oldest first. The id on the left is what the dropdown below
passes along.

In [ ]:
cat = build_catalogue()

print(f"{len(cat)} game(s) across {cat['session'].nunique()} session(s)")
cat[["day", "time", "who", "mode", "hand",
     "trials", "hit_rate", "status"]].rename_axis("id")

## Choose what to analyse

Click one row. Three scopes: one game, one session (a person on one day), or one
person across every day they played.

Choosing only sets the selection. Run the cells below afterwards to see it, and
run them again after changing the pick. Until you choose anything, the newest
game is used.

In [ ]:
import ipywidgets as W
from IPython.display import display

pick = "latest"          # what every cell below analyses

note = W.HTML("<span style='color:#64748b'>newest game until you "
              "choose</span>")


def chosen(change):
    global pick
    if change["new"] is None:            # a heading row, not a save
        note.value = ("<span style='color:#b45309'>that row is a heading, "
                      "choose a save under it</span>")
        return
    pick = change["new"]
    note.value = (f"<span style='color:#16a34a'>selected {pick!r}. Run the "
                  f"cells below.</span>")


menu = W.Dropdown(options=menu_options(cat), value=None,
                  description="Analyse:", layout=W.Layout(width="660px"),
                  style={"description_width": "70px"})
menu.observe(chosen, names="value")
display(W.VBox([menu, note]))

## Load the selection

Reads the trials, the metadata and the calibration once, and adds the calibrated
force columns. Everything below works off what this cell builds, so run it again
after picking something else.

In [ ]:
ctx = prepare(pick)

cat = ctx["cat"]
sel = ctx["sel"]
folders = ctx["folders"]
metas = ctx["metas"]
sessions = ctx["sessions"]
trials = ctx["trials"]
unit = ctx["unit"]
calset = ctx["calset"]
on_task = 0.0            # seconds, filled in by the overview cell below

print(f"{len(folders)} game(s), {len(trials)} trials, "
      f"{sel['session'].nunique()} session(s)")
print(f"force logged in {unit}, calibration: {calset.status}")
sel[["day", "time", "who", "mode", "hand", "trials"]].rename_axis("id")

## Calibration this data was recorded under

What one light press was worth on each pad on the day, which is the thing that
makes force comparable between fingers. One block per distinct calibration, and
a plain statement when a game recorded none.

In [ ]:
cal_tables = sec_calibration(metas, sessions)

## Overview

One row per game, then time on task with pauses taken out. `on_task` is reused
by the dose cell further down.

In [ ]:
on_task = sec_overview(trials, folders, metas)

## Data quality

Cues that never reached the device, how many trials carry force, pauses, and
sensor drift worth looking at.

In [ ]:
sec_quality(trials, folders, metas)

## Comparing the games

Hit rate, speed and consistency side by side. Prints nothing when the selection
holds a single game.

In [ ]:
comparison = sec_compare(trials)

## Reaction time

Cued modes only, misses removed. Distribution, per finger, and the trend across
the block.

In [ ]:
rt = sec_reaction_time(trials)

## Accuracy and the challenge point

Hit rate against the 65 to 80 percent band the adaptive controller aims for,
with missed trials and wrong-finger presses counted separately.

In [ ]:
sec_accuracy(trials)

## Force

Three measures: raw counts as recorded, newtons for the absolute check against
Demouche's healthy data, and force as a fraction of that finger's own
calibration press. Only the last is comparable between fingers, and the cell
states which one it used.

In [ ]:
force = sec_force(trials, unit, calset)

## Finger individuation

Target-finger force over total force. Each lane is divided by its own
calibration press first where one exists, because otherwise an over-reading pad
shows up as spill the hand never produced.

In [ ]:
ind = sec_individuation(trials, calset)

## Rhythm

Beat offsets and whether the tempo was being tracked. Rhythm blocks only.

In [ ]:
rhythm = sec_rhythm(trials)

## Both hands

Left against right. Bilateral blocks only, so nothing prints for a one-handed
selection.

In [ ]:
sec_bilateral(trials, unit, calset)

## Raw sample stream

The 200 Hz log behind the first selected game that has one: press durations and
the average shape of a press.

In [ ]:
sec_raw(folders, unit, calset)

## Movement onset and rate of force development

Onset taken from the force trace rather than from a threshold crossing, and how
far apart the two estimates of reaction time sit.

In [ ]:
onset = sec_onset(folders, trials, unit, calset)

## Objective 1, per-finger hit rate

Each finger against the band over its own 32-trial windows. The session-level
figure can sit inside the band while single fingers sit well outside it.

In [ ]:
objective_one = sec_objective_one(trials, calset=calset)

## Trial exclusions

Trials with no cue delivered and presses faster than 100 ms, with the headline
numbers before and after they come out.

In [ ]:
flagged = sec_exclusions(trials)

## Pretest to aftertest

Only prints once a protocol with phases has been run.

In [ ]:
phases = sec_phase(trials)

## Press thresholds in newtons

What force each finger needed to register a press, against the healthy
fingertip forces Demouche measured. A trigger above those is a threshold
problem, not a weak finger.

In [ ]:
thresholds = sec_threshold_audit(metas=metas)

## Cue modality

Visual, vibration and both compared. Needs blocks recorded under at least two
cue settings.

In [ ]:
cues = sec_cue_modality(trials, calset)

## Dose

Repetitions against Lang's clinical benchmark. Needs `on_task` from the overview
cell above. Without it the per-minute lines are skipped rather than guessed at.

In [ ]:
sec_dose(trials, on_task / 60)

## Sampling

How many logged samples carry new sensor data. That sets the real resolution of
every onset and rate-of-force figure, so it belongs in the limitations.

In [ ]:
sampling = sec_sampling_note(folders)

## Progress per participant

Every session a person has done, in order. This covers everyone on disk, not
just the selection, because the trend across blocks is the outcome measure.

In [ ]:
progress = sec_participant_progress(cat=cat)

## Headline numbers

Built from what the cells above returned, so nothing is recomputed and nothing
can drift from what was printed.

In [ ]:
summary = {"games": len(folders), "trials": len(trials),
           "time_on_task_min": round(on_task / 60, 1),
           "calibration": calset.status}

cued = trials[trials["mode"].isin(["classic", "adaptive", "mirror"])]
if not cued.empty:
    summary["hit_rate"] = round((cued["early_late"] != "Miss").mean(), 3)

if not rt.empty:
    v = rt["time_difference_ms"]
    summary["rt_mean_ms"] = round(v.mean(), 1)
    summary["rt_cv"] = round(v.std() / v.mean(), 3)

if not force.empty:
    summary["force_unit"] = unit
    summary["peak_force_mean_raw"] = round(force["peak_force_n"].mean(), 1)
    summary["peak_force_mean_N"] = round(force["peak_force_N"].mean(), 2)
    if force["peak_force_cal"].notna().any():
        summary["peak_force_mean_cal"] = round(
            force["peak_force_cal"].mean(), 3)

if not ind.empty:
    # Name the basis. A raw index and a calibrated one are different
    # numbers, and only the calibrated one can be read between fingers.
    if ind["individuation_cal"].notna().any():
        summary["individuation_mean"] = round(
            ind["individuation_cal"].mean(), 3)
        summary["individuation_basis"] = "calibrated"
    else:
        summary["individuation_mean"] = round(ind["individuation"].mean(), 3)
        summary["individuation_basis"] = "raw, not corrected"

if not rhythm.empty:
    summary["beat_accuracy_ms"] = round(
        rhythm["time_difference_ms"].abs().mean(), 1)

if not onset.empty:
    summary["onset_rt_mean_ms"] = round(onset["onset_rt_ms"].mean(), 1)
    summary["rfd_mean_raw"] = round(onset["peak_dforce"].mean(), 1)
    if onset["peak_dforce_cal"].notna().any():
        summary["rfd_mean_cal"] = round(onset["peak_dforce_cal"].mean(), 3)

pd.DataFrame([summary]).T.rename(columns={0: "value"})

## Save the tables

Writes the CSVs next to this notebook. Skip this cell to leave the files on disk
as they are.

In [ ]:
written = ["session_summary.csv", "selected_trials.csv"]
pd.DataFrame([summary]).T.rename(columns={0: "value"}).to_csv(
    "session_summary.csv")
trials.to_csv("selected_trials.csv", index=False)
if not ind.empty:
    ind.to_csv("individuation_per_trial.csv", index=False)
    written.append("individuation_per_trial.csv")

print("written next to this notebook: " + ", ".join(written))
print("figures are in figures/ at 160 dpi, sized for the report")